# Weather API

In [2]:
"""
OPEN-METEO API
      ↓
Daily weather data
      ↓
Clean/prepare dates
      ↓
Convert daily weather
into monthly weather
      ↓
One weather row per month
      ↓
Combine with Airbnb monthly data
      ↓
Analyze H6:
Does weather affect Airbnb occupancy/revenue?
"""

'\nOPEN-METEO API\n      ↓\nDaily weather data\n      ↓\nClean/prepare dates\n      ↓\nConvert daily weather\ninto monthly weather\n      ↓\nOne weather row per month\n      ↓\nCombine with Airbnb monthly data\n      ↓\nAnalyze H6:\nDoes weather affect Airbnb occupancy/revenue?\n'

In [3]:
# Import pandas to work with our weather data as a DataFrame
import pandas as pd

# Import requests to send a request to the Open-Meteo API
import requests

In [4]:
# This is the Open-Meteo historical weather API endpoint
url = "https://archive-api.open-meteo.com/v1/archive"

# Define the parameters we want to send to the API
params = {
    # Coordinates for Barcelona
    "latitude": 41.3874,
    "longitude": 2.1686,
    # First date we want
    "start_date": "2021-03-01",
    # Last date we want
    "end_date": "2026-02-28",
    # Weather variables we want from the API
    "daily": "temperature_2m_mean,precipitation_sum,sunshine_duration",
    # Use Barcelona's local time zone
    "timezone": "Europe/Madrid",
}

# Send our request to the Open-Meteo API
response = requests.get(url, params=params)

# Check whether the request was successful
response.status_code

200

In [5]:
# Convert the API response from JSON format into a Python dictionary
weather_data = response.json()

# Check the main sections contained in the API response
weather_data.keys()

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])

In [6]:
weather_data["daily"].keys()

dict_keys(['time', 'temperature_2m_mean', 'precipitation_sum', 'sunshine_duration'])

In [7]:
weather_data["daily"]

# Show first 5 values for each column
for key, values in weather_data["daily"].items():
    print(key, values[:5])

time ['2021-03-01', '2021-03-02', '2021-03-03', '2021-03-04', '2021-03-05']
temperature_2m_mean [12.3, 12.5, 11.5, 11.1, 11.1]
precipitation_sum [0.3, 0.5, 0.9, 0.0, 0.0]
sunshine_duration [36667.64, 31433.09, 30573.98, 39281.94, 37999.71]


In [8]:
# Create df
weather_df = pd.DataFrame(weather_data["daily"])

weather_df.head()

,time,temperature_2m_mean,precipitation_sum,sunshine_duration
0,2021-03-01,12.3,0.3,36667.64
1,2021-03-02,12.5,0.5,31433.09
2,2021-03-03,11.5,0.9,30573.98
3,2021-03-04,11.1,0.0,39281.94
4,2021-03-05,11.1,0.0,37999.71


In [9]:
weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1826 entries, 0 to 1825
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   time                 1826 non-null   str    
 1   temperature_2m_mean  1826 non-null   float64
 2   precipitation_sum    1826 non-null   float64
 3   sunshine_duration    1826 non-null   float64
dtypes: float64(3), str(1)
memory usage: 75.0 KB


In [10]:
# Step 1: Convert time to datetime
weather_df["time"] = pd.to_datetime(weather_df["time"])

# Check the number of rows, columns, data types and missing values
weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1826 entries, 0 to 1825
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   time                 1826 non-null   datetime64[us]
 1   temperature_2m_mean  1826 non-null   float64       
 2   precipitation_sum    1826 non-null   float64       
 3   sunshine_duration    1826 non-null   float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 57.2 KB


In [11]:
# Step 2: Convert monthly date format to extract year/month/day
# Convert the 'time' column from text into a real datetime column
# For example, every day in March 2021 becomes 2021-03-01
weather_df["month_date"] = weather_df["time"].dt.to_period("M").dt.to_timestamp()

weather_df.head()

,time,temperature_2m_mean,precipitation_sum,sunshine_duration,month_date
0,2021-03-01,12.3,0.3,36667.64,2021-03-01
1,2021-03-02,12.5,0.5,31433.09,2021-03-01
2,2021-03-03,11.5,0.9,30573.98,2021-03-01
3,2021-03-04,11.1,0.0,39281.94,2021-03-01
4,2021-03-05,11.1,0.0,37999.71,2021-03-01


In [12]:
# Group the daily weather observations by month.
# Calculate monthly value for each weather variable.
weather_monthly = (
    weather_df.groupby(
        "month_date"
    )  # Put all days belonging to the same month together.
    .agg(
        avg_temperature=(
            "temperature_2m_mean",
            "mean",
        ),  # Calculate average temperature for that month.
        total_precipitation=(
            "precipitation_sum",
            "sum",
        ),  # Add up all precipitation during that month.
        total_sunshine=(
            "sunshine_duration",
            "sum",
        ),  # Add up all sunshine duration during that month.
    )
    .reset_index()
)

# Display the first few months
weather_monthly.head()

,month_date,avg_temperature,total_precipitation,total_sunshine
0,2021-03-01,11.119355,22.8,1174759.16
1,2021-04-01,12.880000,38.6,1058421.49
2,2021-05-01,16.948387,36.2,1312771.51
3,2021-06-01,22.776667,25.5,1415663.76
4,2021-07-01,24.667742,16.8,1521023.19


In [13]:
# Convert total sunshine from seconds to hours.
weather_monthly["sunshine_hours"] = weather_monthly["total_sunshine"] / 3600

# Check the result
weather_monthly.head()

,month_date,avg_temperature,total_precipitation,total_sunshine,sunshine_hours
0,2021-03-01,11.119355,22.8,1174759.16,326.321989
1,2021-04-01,12.880000,38.6,1058421.49,294.005969
2,2021-05-01,16.948387,36.2,1312771.51,364.658753
3,2021-06-01,22.776667,25.5,1415663.76,393.239933
4,2021-07-01,24.667742,16.8,1521023.19,422.506442


In [14]:
weather_monthly.shape

# 60 months: March 2021 -> Feb 2026

(60, 5)

In [15]:
weather_monthly.head()

,month_date,avg_temperature,total_precipitation,total_sunshine,sunshine_hours
0,2021-03-01,11.119355,22.8,1174759.16,326.321989
1,2021-04-01,12.880000,38.6,1058421.49,294.005969
2,2021-05-01,16.948387,36.2,1312771.51,364.658753
3,2021-06-01,22.776667,25.5,1415663.76,393.239933
4,2021-07-01,24.667742,16.8,1521023.19,422.506442


In [16]:
weather_monthly.tail()

,month_date,avg_temperature,total_precipitation,total_sunshine,sunshine_hours
55,2025-10-01,18.312903,57.4,1017497.72,282.638256
56,2025-11-01,12.873333,19.7,870462.80,241.795222
57,2025-12-01,10.641935,161.0,704389.54,195.663761
58,2026-01-01,9.032258,139.1,682055.40,189.459833
59,2026-02-01,11.971429,38.5,831709.42,231.030394


In [17]:
# Save the monthly weather dataset as a CSV file
weather_monthly.to_csv("weather_final.csv", index=False)